# 04 - Enrich (language, experience, skills, role, category)

| Stage | Runs on | Method |
|---|---|---|
| 1. Rules | Every new / changed job (`job_key` + `description_hash` not yet enriched with this `enrichment_version`) | `langdetect`, regex experience + level, `ops.ref_skills` taxonomy, title keyword rules (category hint) |
| 2. Roles | Gold-eligible jobs (India + English) without a role, or with an ambiguous one | Title + description and every `ops.ref_roles` role are embedded with `embedding_endpoint` (gte-large) via `ai_query`, cached in `ops.embedding_cache`. Cosine similarity + small bonuses (role alias in title, same category as the title rule). Close calls go to `role_llm_endpoint` (70B), which picks from the top 5 candidates. **Category = the role's category** |
| 3. Skills LLM | Gold-eligible jobs with fewer than `min_skills` skills on a real description | `ai_query` structured JSON, grounded in the job text, cached in `ops.enrichment_llm_log` |

All model calls are capped per run (`embedding_max_rows`, `role_llm_max_rows`, `llm_max_rows`) and cached, so the backlog is worked off over several runs and nothing is ever paid for twice.

`role_method`: `embedding` / `embedding_ambiguous` (best embedding role kept, LLM tie-break pending) / `llm`.
`llm_status` (skills): `not_needed` / `pending` / `done` / `failed` / `skipped`.

**Job task environment dependencies:** `langdetect==1.0.9` (`model2vec` is no longer used)

In [ ]:
%run ./_common

In [ ]:
import json
import re

import numpy as np
from langdetect import DetectorFactory, LangDetectException, detect_langs
from pyspark.sql import Window
from pyspark.sql import functions as F

DetectorFactory.seed = 0

dbutils.widgets.text("enrichment_version", "v4", "Enrichment version (bump to re-run rules on everything)")
dbutils.widgets.dropdown("force_reenrich", "false", ["false", "true"], "Force re-enrich all silver rows")
dbutils.widgets.dropdown("force_rematch_roles", "false", ["false", "true"], "Re-score roles for all eligible jobs (uses caches)")
dbutils.widgets.text("min_skills", "3", "Fewer skills than this -> skills LLM")
dbutils.widgets.text("embedding_endpoint", "databricks-gte-large-en", "Embedding endpoint")
dbutils.widgets.text("embedding_max_rows", "600", "Max new job embeddings per run")
dbutils.widgets.text("role_min_margin", "0.02", "Best - 2nd role score below this -> LLM tie-break")
dbutils.widgets.text("role_llm_endpoint", "databricks-meta-llama-3-3-70b-instruct", "Role tie-break LLM endpoint")
dbutils.widgets.text("role_llm_max_rows", "200", "Max new role tie-break LLM calls per run")
dbutils.widgets.dropdown("llm_enabled", "true", ["true", "false"], "Use LLMs (skills + role tie-break)")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-1-8b-instruct", "Skills LLM endpoint")
dbutils.widgets.text("llm_max_rows", "300", "Max new skills LLM calls per run")

ENRICHMENT_VERSION = dbutils.widgets.get("enrichment_version").strip()
FORCE_REENRICH = dbutils.widgets.get("force_reenrich") == "true"
FORCE_REMATCH_ROLES = dbutils.widgets.get("force_rematch_roles") == "true"
MIN_SKILLS = int(dbutils.widgets.get("min_skills"))
EMBEDDING_ENDPOINT = dbutils.widgets.get("embedding_endpoint").strip()
EMBEDDING_MAX_ROWS = int(dbutils.widgets.get("embedding_max_rows"))
ROLE_MIN_MARGIN = float(dbutils.widgets.get("role_min_margin"))
ROLE_LLM_ENDPOINT = dbutils.widgets.get("role_llm_endpoint").strip()
ROLE_LLM_MAX_ROWS = int(dbutils.widgets.get("role_llm_max_rows"))
LLM_ENABLED = dbutils.widgets.get("llm_enabled") == "true"
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint").strip()
LLM_MAX_ROWS = int(dbutils.widgets.get("llm_max_rows"))

LLM_MAX_CHARS = 4000          # description characters sent to the skills LLM
JOB_TEXT_CHARS = 2000         # title + description characters embedded / sent for role tie-break
TITLE_ALIAS_BONUS = 0.05      # role alias found in the job title
CATEGORY_RULE_BONUS = 0.03    # role category equals the title keyword category
ROLE_CANDIDATES = 5
PROMPT_VERSION = "p2"         # skills prompt; bump when LLM_PROMPT or RESPONSE_FORMAT change
ROLE_PROMPT_VERSION = "r1"    # role tie-break prompt; bump when ROLE_PROMPT changes

for name, value in [("embedding_endpoint", EMBEDDING_ENDPOINT), ("role_llm_endpoint", ROLE_LLM_ENDPOINT),
                    ("llm_endpoint", LLM_ENDPOINT), ("enrichment_version", ENRICHMENT_VERSION)]:
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", value):
        raise ValueError(f"Invalid {name}: {value!r}")

SILVER_TABLE = table("silver", "jobs_clean")
ENRICHED_TABLE = table("silver", "jobs_enriched")
LLM_LOG_TABLE = table("ops", "enrichment_llm_log")
EMBEDDING_CACHE_TABLE = table("ops", "embedding_cache")
ROLES_TABLE = table("ops", "ref_roles")

print(f"version={ENRICHMENT_VERSION} embeddings={EMBEDDING_ENDPOINT} max={EMBEDDING_MAX_ROWS} "
      f"role_llm={ROLE_LLM_ENDPOINT}/{ROLE_PROMPT_VERSION} max={ROLE_LLM_MAX_ROWS} "
      f"skills_llm={LLM_ENDPOINT}/{PROMPT_VERSION} max={LLM_MAX_ROWS} llm_enabled={LLM_ENABLED}")

In [ ]:
# Schema guard: objects added to 01_setup after its first run. Keeps this notebook runnable even if setup wasn't re-run.
add_missing_columns(ENRICHED_TABLE, {
    "llm_status": "STRING", "role_title": "STRING", "role_score": "DOUBLE", "role_alternative": "STRING",
    "role_alternative_score": "DOUBLE", "role_method": "STRING",
})

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LLM_LOG_TABLE} (
  job_key STRING, description_hash STRING, endpoint STRING, prompt_version STRING,
  response STRING, error STRING, called_at TIMESTAMP, _job_run_id STRING
)
""")
add_missing_columns(LLM_LOG_TABLE, {"prompt_version": "STRING"})

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {EMBEDDING_CACHE_TABLE} (
  text_hash STRING NOT NULL, model STRING NOT NULL, cache_key STRING, embedding ARRAY<FLOAT>,
  error STRING, created_at TIMESTAMP, _job_run_id STRING
)
""")

if not spark.catalog.tableExists(ROLES_TABLE):
    raise RuntimeError(f"{ROLES_TABLE} not found - re-import and run 01_setup first")

## Reference data and rules

In [ ]:
categories = spark.table(table("ops", "ref_categories")).orderBy("priority").collect()
skills_ref = spark.table(table("ops", "ref_skills")).collect()
roles = spark.table(ROLES_TABLE).orderBy("category", "role_title").collect()
CATEGORY_NAMES = [row["category"] for row in categories]
ROLE_TITLES = [row["role_title"] for row in roles]
ROLE_CATEGORY = {row["role_title"]: row["category"] for row in roles}


def keyword_regex(keywords):
    alternatives = sorted({keyword.lower().strip() for keyword in keywords if keyword.strip()}, key=len, reverse=True)
    return re.compile(r"(?<![a-z0-9])(?:" + "|".join(re.escape(a) for a in alternatives) + r")(?![a-z0-9])")


TITLE_RULES = [(row["category"], keyword_regex(row["title_keywords"])) for row in categories if row["title_keywords"]]
ROLE_ALIAS_RULES = [keyword_regex(row["aliases"]) if row["aliases"] else None for row in roles]


def clean_title(title):
    return re.sub(r"_+", " ", title or "")  # Workday titles like IN_Senior Associate_Data Engineering


def title_category(title):
    lowered = clean_title(title).lower()
    for category, pattern in TITLE_RULES:
        if pattern.search(lowered):
            return category
    return None


# Skills: one regex over all aliases, longest first. A few common words need negative context.
ALIAS_TO_SKILL = {}
for row in skills_ref:
    for alias in row["aliases"] or []:
        ALIAS_TO_SKILL.setdefault(alias.lower(), (row["skill"], row["skill_group"]))

NEGATIVE_CONTEXT = {
    "excel": r"(?!\s+(?:in|at|as)\b)",
    "spark": r"(?!\s+(?:greatness|joy|curiosity|creativity|innovation|conversations?|change|ideas?)\b)",
    "agentic": r"(?!\s+(?:marketing|commerce)\b)",  # WPP boilerplate: "agentic marketing platform"
}
SKILL_RE = re.compile(
    r"(?<![a-z0-9])("
    + "|".join(re.escape(alias) + NEGATIVE_CONTEXT.get(alias, "") for alias in sorted(ALIAS_TO_SKILL, key=len, reverse=True))
    + r")(?![a-z0-9])",
    re.IGNORECASE,
)


def extract_skills(text):
    skills, groups = [], []
    for match in SKILL_RE.finditer(text or ""):
        skill, group = ALIAS_TO_SKILL[match.group(1).lower()]
        if skill not in skills:
            skills.append(skill)
        if group not in groups:
            groups.append(group)
    return skills, groups


# Experience
YEARS = r"(?:years?|yrs?)"
EXPERIENCE_PATTERNS = [
    ("range", re.compile(rf"(\d{{1,2}})\s*\+?\s*(?:-|–|—|to)\s*(\d{{1,2}})\s*\+?\s*{YEARS}", re.I)),
    ("min", re.compile(rf"(?:minimum|min\.?|at least|over|more than)\s*(?:of\s*)?(\d{{1,2}})\s*\+?\s*{YEARS}", re.I)),
    ("plus", re.compile(rf"(\d{{1,2}})\s*(?:\+|plus)\s*{YEARS}", re.I)),
    ("plain", re.compile(rf"(\d{{1,2}})\s*{YEARS}", re.I)),
]
EXPERIENCE_WORD = re.compile(r"experience|\bexp\b|\bexp\.", re.I)
EXCLUDE_CONTEXT = re.compile(r"education|degree|years? old|\bage\b|anniversary|founded|history|warranty|since \d{4}", re.I)
FRESHER_RE = re.compile(r"\bfreshers?\b|\bno (?:prior )?experience (?:is )?required\b", re.I)


def extract_experience(text):
    text = text or ""
    candidates = []
    for priority, (kind, pattern) in enumerate(EXPERIENCE_PATTERNS):
        for match in pattern.finditer(text):
            window = text[max(0, match.start() - 80): match.end() + 80]
            close = text[max(0, match.start() - 30): match.end() + 30]
            if not EXPERIENCE_WORD.search(window) or EXCLUDE_CONTEXT.search(close):
                continue
            low = int(match.group(1))
            high = int(match.group(2)) if kind == "range" else None
            if high is not None and high < low:
                low, high = high, low
            if low > 30 or (high is not None and high > 40):
                continue
            candidates.append((match.start(), priority, low, high, match.group(0)))
    if candidates:
        _, _, low, high, snippet = min(candidates)
        return low, high, snippet.strip()
    if FRESHER_RE.search(text):
        return 0, 1, "fresher"
    return None, None, None


LEVEL_RULES = [
    ("Intern", re.compile(r"\b(?:intern|internship|trainee|apprentice)\b", re.I)),
    ("Director+", re.compile(r"\b(?:director|vp|vice president|head of|chief|cto|cio|cdo)\b", re.I)),
    ("Principal/Staff", re.compile(r"\b(?:principal|staff|distinguished|architect)\b", re.I)),
    ("Lead/Manager", re.compile(r"\b(?:lead|manager|supervisor)\b", re.I)),
    ("Senior", re.compile(r"\b(?:senior|sr\b\.?|iii|iv)\b", re.I)),
    ("Junior", re.compile(r"\b(?:junior|jr\b\.?|associate|graduate|fresher|entry)\b", re.I)),
]
LINKEDIN_LEVELS = {"internship": "Intern", "entry level": "Entry", "associate": "Junior", "mid-senior level": "Mid-Senior",
                   "director": "Director+", "executive": "Director+"}


def years_level(min_years):
    if min_years is None:
        return None
    if min_years < 1:
        return "Entry"
    if min_years < 3:
        return "Junior"
    if min_years < 6:
        return "Mid"
    if min_years < 10:
        return "Senior"
    return "Lead/Manager"


def experience_level(title, seniority_level, min_years):
    for level, pattern in LEVEL_RULES:
        if pattern.search(title or ""):
            return level
    return years_level(min_years) or LINKEDIN_LEVELS.get((seniority_level or "").lower())


def detect_language(text):
    sample = (text or "")[:1500]
    if len(sample) < 100:
        return None, None  # too short to be reliable -> treated as unknown (not excluded)
    try:
        best = detect_langs(sample)[0]
        return best.lang, round(float(best.prob), 4)
    except LangDetectException:
        return None, None


print(f"{len(TITLE_RULES)} title rules, {len(ALIAS_TO_SKILL)} skill aliases, {len(ROLE_TITLES)} roles")
print(extract_experience("We need 5+ years of experience with Spark and 3-5 years in Azure."))

## Stage 1 - rules on new / changed jobs

In [ ]:
ENRICHED_SCHEMA = (
    "job_key STRING, description_hash STRING, language STRING, language_confidence DOUBLE, "
    "experience_min_years INT, experience_max_years INT, experience_text STRING, experience_level STRING, "
    "skills ARRAY<STRING>, skill_groups ARRAY<STRING>, category STRING, category_score DOUBLE, category_method STRING, "
    "gold_eligible BOOLEAN, gold_exclusion_reason STRING, llm_status STRING, "
    "role_title STRING, role_score DOUBLE, role_alternative STRING, role_alternative_score DOUBLE, role_method STRING, "
    "enrichment_version STRING, enriched_at TIMESTAMP, _job_run_id STRING"
)
ENRICHED_COLUMNS = [part.strip().split(" ")[0] for part in ENRICHED_SCHEMA.split(", ")]


def enrich_rows(rows):
    """Rule-based enrichment. Role columns are reset to NULL; stage 2 fills them for eligible jobs."""
    tuples = []
    for row in rows:
        title = clean_title(row["title"])
        description = row["description"] or ""

        language, language_confidence = detect_language(description)
        min_years, max_years, experience_text = extract_experience(description)
        skills, skill_groups = extract_skills(f"{title}. {description}")
        category = title_category(title)

        exclusion = []
        if not row["is_india"]:
            exclusion.append("not_india")
        if language not in (None, "en"):
            exclusion.append(f"non_english:{language}")
        eligible = not exclusion

        few_skills = len(skills) < MIN_SKILLS and len(description) >= 300
        output = {
            "job_key": row["job_key"],
            "description_hash": row["description_hash"],
            "language": language,
            "language_confidence": language_confidence,
            "experience_min_years": min_years,
            "experience_max_years": max_years,
            "experience_text": experience_text,
            "experience_level": experience_level(title, row["seniority_level"], min_years),
            "skills": skills,
            "skill_groups": skill_groups,
            "category": category,  # title-rule hint; replaced by the role's category in stage 2
            "category_score": 1.0 if category else None,
            "category_method": "title_rule" if category else None,
            "gold_eligible": eligible,
            "gold_exclusion_reason": ",".join(exclusion) or None,
            "llm_status": "skipped" if not eligible else ("pending" if few_skills else "not_needed"),
            "role_title": None, "role_score": None, "role_alternative": None, "role_alternative_score": None, "role_method": None,
            "enrichment_version": ENRICHMENT_VERSION,
            "enriched_at": None,
            "_job_run_id": JOB_RUN_ID,
        }
        tuples.append(tuple(output[column] for column in ENRICHED_COLUMNS))
    return tuples

In [ ]:
with task_run("enrich") as metrics:
    silver = spark.table(SILVER_TABLE).select("job_key", "description_hash", "title", "description", "is_india", "seniority_level")
    if FORCE_REENRICH:
        pending = silver
    else:
        already = spark.table(ENRICHED_TABLE).where(F.col("enrichment_version") == ENRICHMENT_VERSION).select("job_key", "description_hash")
        pending = silver.join(already, ["job_key", "description_hash"], "left_anti")

    pending_rows = pending.collect()
    enriched_tuples = enrich_rows(pending_rows)
    metrics["rows_in"] = len(pending_rows)

    if enriched_tuples:
        (
            spark.createDataFrame(enriched_tuples, ENRICHED_SCHEMA)
            .withColumn("enriched_at", F.current_timestamp())
            .createOrReplaceTempView("enriched_updates")
        )
        spark.sql(f"""
            MERGE INTO {ENRICHED_TABLE} t
            USING enriched_updates s
            ON t.job_key = s.job_key
            WHEN MATCHED THEN UPDATE SET {", ".join(f"t.{c} = s.{c}" for c in ENRICHED_COLUMNS if c != "job_key")}
            WHEN NOT MATCHED THEN INSERT ({", ".join(ENRICHED_COLUMNS)}) VALUES ({", ".join(f"s.{c}" for c in ENRICHED_COLUMNS)})
        """)

    status_index = ENRICHED_COLUMNS.index("llm_status")
    metrics["details"]["stage1"] = {
        "rows": len(enriched_tuples),
        "eligible": sum(1 for t in enriched_tuples if t[ENRICHED_COLUMNS.index("gold_eligible")]),
        "llm_status": {s: sum(1 for t in enriched_tuples if t[status_index] == s) for s in {t[status_index] for t in enriched_tuples}},
    }
    print(f"Stage 1 enriched {len(enriched_tuples)} rows: {metrics['details']['stage1']}")

## Stage 2 - role matching (large embeddings + LLM tie-break)

In [ ]:
def ai_result_fields(df, column):
    """Field names of an ai_query failOnError=>false struct, resolved from the schema instead of hard-coded."""
    fields = df.schema[column].dataType.fieldNames()
    result_field = "result" if "result" in fields else fields[0]
    error_field = next((name for name in fields if "error" in name.lower()), None)
    return result_field, error_field


def embed_missing(texts_df, max_rows):
    """Embeds texts (cache_key, text, text_hash) not yet cached for this model. Returns (new calls, failures)."""
    cached = (
        spark.table(EMBEDDING_CACHE_TABLE)
        .where((F.col("model") == EMBEDDING_ENDPOINT) & F.col("embedding").isNotNull())
        .select("text_hash").distinct()
    )
    todo = texts_df.join(cached, "text_hash", "left_anti").dropDuplicates(["text_hash"])
    if max_rows is not None:
        todo = todo.limit(max_rows)
    # Materialise the batch first so the billable calls run on a fixed set of texts exactly once
    todo_rows = todo.select("cache_key", "text", "text_hash").collect()
    if not todo_rows:
        return 0, 0

    responses = spark.createDataFrame(todo_rows, "cache_key STRING, text STRING, text_hash STRING").select(
        "text_hash", "cache_key",
        F.expr(f"ai_query('{EMBEDDING_ENDPOINT}', text, failOnError => false)").alias("r"),
    )
    result_field, error_field = ai_result_fields(responses, "r")
    (
        responses.select(
            "text_hash", F.lit(EMBEDDING_ENDPOINT).alias("model"), "cache_key",
            F.col(f"r.{result_field}").cast("array<float>").alias("embedding"),
            (F.col(f"r.{error_field}").cast("string") if error_field else F.lit(None).cast("string")).alias("error"),
            F.current_timestamp().alias("created_at"), F.lit(JOB_RUN_ID).alias("_job_run_id"),
        )
        .write.mode("append").saveAsTable(EMBEDDING_CACHE_TABLE)
    )
    failures = (
        spark.table(EMBEDDING_CACHE_TABLE)
        .where((F.col("_job_run_id") == JOB_RUN_ID) & (F.col("model") == EMBEDDING_ENDPOINT) & F.col("embedding").isNull())
        .where(F.col("text_hash").isin([row["text_hash"] for row in todo_rows]))
        .count()
    )
    return len(todo_rows), failures


def cached_embeddings():
    latest = Window.partitionBy("text_hash").orderBy(F.col("created_at").desc())
    return (
        spark.table(EMBEDDING_CACHE_TABLE)
        .where((F.col("model") == EMBEDDING_ENDPOINT) & F.col("embedding").isNotNull())
        .withColumn("_rank", F.row_number().over(latest)).where("_rank = 1")
        .select("text_hash", "embedding")
    )


def normalize(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.where(norms == 0, 1, norms)


ROLE_PROMPT = (
    "Choose the standard role that best matches this job posting. Consider the responsibilities and required skills in the "
    "description, not only the title. Return JSON with role_title set to exactly one of the candidate role titles.\n"
)
ROLE_RESPONSE_FORMAT = json.dumps({
    "type": "json_schema",
    "json_schema": {
        "name": "role_choice",
        "schema": {
            "type": "object",
            "properties": {"role_title": {"type": "string", "enum": ROLE_TITLES}},
            "required": ["role_title"],
            "additionalProperties": False,
        },
        "strict": True,
    },
})
assert "'" not in ROLE_RESPONSE_FORMAT, "role response format must not contain single quotes"
ROLE_DESCRIPTIONS = {row["role_title"]: row["description"] for row in roles}
ROLE_ALIAS_BY_TITLE = dict(zip(ROLE_TITLES, ROLE_ALIAS_RULES))

In [ ]:
role_counts = {}

with task_run("enrich_roles") as metrics:
    # Roles: embed once (cached forever unless the role text changes)
    role_texts = spark.createDataFrame(
        [(f"role:{row['role_title']}", f"{row['role_title']}. {row['description']}") for row in roles],
        "cache_key STRING, text STRING",
    ).withColumn("text_hash", F.sha2("text", 256))
    new_role_embeddings, role_embedding_failures = embed_missing(role_texts, None)

    # Jobs that need a role: eligible and (no role yet, ambiguous, or forced)
    role_filter = "TRUE" if FORCE_REMATCH_ROLES else "(e.role_method IS NULL OR e.role_method = 'embedding_ambiguous')"
    need = spark.sql(f"""
        SELECT e.job_key, e.description_hash, s.title,
               substring(concat_ws('. ', s.title, coalesce(s.description, '')), 1, {JOB_TEXT_CHARS}) AS text
        FROM {ENRICHED_TABLE} e
        JOIN {SILVER_TABLE} s ON s.job_key = e.job_key AND s.description_hash = e.description_hash
        WHERE e.gold_eligible AND {role_filter}
    """).withColumn("text_hash", F.sha2("text", 256))

    new_job_embeddings, job_embedding_failures = embed_missing(
        need.select(F.col("job_key").alias("cache_key"), "text", "text_hash"), EMBEDDING_MAX_ROWS
    )

    embeddings = cached_embeddings()
    role_vectors = {
        row["cache_key"][len("role:"):]: row["embedding"]
        for row in role_texts.join(embeddings, "text_hash").select("cache_key", "embedding").collect()
    }
    usable_roles = [row for row in roles if row["role_title"] in role_vectors]
    if len(usable_roles) < 2:
        raise RuntimeError(f"Only {len(usable_roles)} role embeddings available - check the embedding endpoint {EMBEDDING_ENDPOINT}")
    role_matrix = normalize(np.array([role_vectors[row["role_title"]] for row in usable_roles], dtype=np.float32))
    role_alias_rules = [ROLE_ALIAS_BY_TITLE[row["role_title"]] for row in usable_roles]
    role_categories = [row["category"] for row in usable_roles]

    job_rows = need.join(embeddings, "text_hash").select("job_key", "description_hash", "title", "text", "embedding").collect()

    scored = {}
    if job_rows:
        job_matrix = normalize(np.array([row["embedding"] for row in job_rows], dtype=np.float32))
        similarity = job_matrix @ role_matrix.T
        for index, row in enumerate(job_rows):
            title = clean_title(row["title"]).lower()
            rule_category = title_category(row["title"])  # recomputed: e.category may already hold a previous role's category
            scores = similarity[index].astype(float).copy()
            for role_index, alias_rule in enumerate(role_alias_rules):
                if alias_rule is not None and alias_rule.search(title):
                    scores[role_index] += TITLE_ALIAS_BONUS
                if rule_category and role_categories[role_index] == rule_category:
                    scores[role_index] += CATEGORY_RULE_BONUS
            order = np.argsort(scores)[::-1]
            best, second = usable_roles[order[0]]["role_title"], usable_roles[order[1]]["role_title"]
            ambiguous = (scores[order[0]] - scores[order[1]]) < ROLE_MIN_MARGIN
            scored[(row["job_key"], row["description_hash"])] = {
                "role_title": best, "role_score": round(float(scores[order[0]]), 4),
                "role_alternative": second, "role_alternative_score": round(float(scores[order[1]]), 4),
                "role_method": "embedding_ambiguous" if ambiguous else "embedding",
                "candidates": [usable_roles[i]["role_title"] for i in order[:ROLE_CANDIDATES]],
                "text": row["text"],
            }

    # LLM tie-break for ambiguous jobs: cached answers first, then new calls up to the cap
    ambiguous_keys = [key for key, value in scored.items() if value["role_method"] == "embedding_ambiguous"]
    role_llm_calls = role_llm_failures = role_llm_cache_hits = 0
    if LLM_ENABLED and ambiguous_keys:
        this_prompt = (F.col("endpoint") == ROLE_LLM_ENDPOINT) & (F.col("prompt_version") == ROLE_PROMPT_VERSION)
        ambiguous_df = spark.createDataFrame(ambiguous_keys, "job_key STRING, description_hash STRING")

        def cached_answers():
            latest = Window.partitionBy("job_key", "description_hash").orderBy(F.col("called_at").desc())
            return {
                (row["job_key"], row["description_hash"]): (row["answer"], row["_job_run_id"])
                for row in spark.table(LLM_LOG_TABLE)
                .where(this_prompt & F.col("error").isNull() & F.col("response").isNotNull())
                .join(ambiguous_df, ["job_key", "description_hash"])
                .withColumn("_rank", F.row_number().over(latest)).where("_rank = 1")
                .select("job_key", "description_hash", "_job_run_id",
                        F.from_json("response", "struct<role_title:string>").role_title.alias("answer"))
                .collect()
            }

        answers = cached_answers()
        to_call = [key for key in ambiguous_keys if key not in answers][:ROLE_LLM_MAX_ROWS]
        if to_call:
            requests = spark.createDataFrame(
                [
                    (
                        key[0], key[1],
                        ROLE_PROMPT
                        + "\nCandidate roles:\n"
                        + "\n".join(f"- {title}: {ROLE_DESCRIPTIONS[title]}" for title in scored[key]["candidates"])
                        + f"\n\nJob posting:\n{scored[key]['text']}",
                    )
                    for key in to_call
                ],
                "job_key STRING, description_hash STRING, request STRING",
            )
            responses = requests.select(
                "job_key", "description_hash",
                F.expr(f"ai_query('{ROLE_LLM_ENDPOINT}', request, responseFormat => '{ROLE_RESPONSE_FORMAT}', failOnError => false)").alias("r"),
            )
            result_field, error_field = ai_result_fields(responses, "r")
            (
                responses.select(
                    "job_key", "description_hash", F.lit(ROLE_LLM_ENDPOINT).alias("endpoint"),
                    F.lit(ROLE_PROMPT_VERSION).alias("prompt_version"),
                    F.col(f"r.{result_field}").cast("string").alias("response"),
                    (F.col(f"r.{error_field}").cast("string") if error_field else F.lit(None).cast("string")).alias("error"),
                    F.current_timestamp().alias("called_at"), F.lit(JOB_RUN_ID).alias("_job_run_id"),
                )
                .write.mode("append").saveAsTable(LLM_LOG_TABLE)
            )
            role_llm_calls = len(to_call)
            answers = cached_answers()
            role_llm_failures = sum(1 for key in to_call if key not in answers)

        for key in ambiguous_keys:
            answer, answer_run = answers.get(key, (None, None))
            if answer and answer in scored[key]["candidates"]:
                if answer_run != JOB_RUN_ID:
                    role_llm_cache_hits += 1
                if answer != scored[key]["role_title"]:
                    scored[key]["role_alternative"] = scored[key]["role_title"]
                    scored[key]["role_alternative_score"] = scored[key]["role_score"]
                    scored[key]["role_title"] = answer
                scored[key]["role_method"] = "llm"

    # Write roles; the job category becomes the role's category
    if scored:
        (
            spark.createDataFrame(
                [
                    (key[0], key[1], value["role_title"], value["role_score"], value["role_alternative"],
                     value["role_alternative_score"], value["role_method"], ROLE_CATEGORY[value["role_title"]])
                    for key, value in scored.items()
                ],
                "job_key STRING, description_hash STRING, role_title STRING, role_score DOUBLE, role_alternative STRING, "
                "role_alternative_score DOUBLE, role_method STRING, role_category STRING",
            )
            .createOrReplaceTempView("role_updates")
        )
        spark.sql(f"""
            MERGE INTO {ENRICHED_TABLE} t
            USING role_updates s
            ON t.job_key = s.job_key AND t.description_hash = s.description_hash
            WHEN MATCHED THEN UPDATE SET
              t.role_title = s.role_title,
              t.role_score = s.role_score,
              t.role_alternative = s.role_alternative,
              t.role_alternative_score = s.role_alternative_score,
              t.role_method = s.role_method,
              t.category = s.role_category,
              t.category_score = s.role_score,
              t.category_method = CASE WHEN s.role_method = 'llm' THEN 'role_llm' ELSE 'role_embedding' END,
              t.enriched_at = current_timestamp(),
              t._job_run_id = '{JOB_RUN_ID}'
        """)

    role_counts = {m: sum(1 for v in scored.values() if v["role_method"] == m) for m in ("embedding", "embedding_ambiguous", "llm")}
    waiting_for_embedding = need.join(embeddings, "text_hash", "left_anti").count()
    metrics.update(rows_in=len(job_rows), rows_out=len(scored), rows_rejected=role_llm_failures + job_embedding_failures)
    metrics["details"].update(
        role_embeddings_new=new_role_embeddings, role_embedding_failures=role_embedding_failures,
        job_embeddings_new=new_job_embeddings, job_embedding_failures=job_embedding_failures,
        waiting_for_embedding=waiting_for_embedding, role_methods=role_counts,
        role_llm_calls=role_llm_calls, role_llm_cache_hits=role_llm_cache_hits, role_llm_failures=role_llm_failures,
    )
    print(f"roles scored={len(scored)} {role_counts} | job embeddings new={new_job_embeddings} failed={job_embedding_failures} "
          f"waiting={waiting_for_embedding} | role LLM calls={role_llm_calls} cache hits={role_llm_cache_hits} failed={role_llm_failures}")

## Stage 3 - skills LLM for gold-eligible jobs with few skills

In [ ]:
LLM_PROMPT = (
    "You extract structured data from a job posting. Return JSON only.\n"
    "- skills: up to 20 tools, technologies, programming languages, platforms, certifications or domain methods "
    "that are written in the job title or description. Copy each name as it appears, 1 to 4 words. "
    "Never add a skill that is not in the text. "
    "Exclude soft skills (communication, teamwork, leadership), generic words (processes, policies, documentation), "
    "benefits, company names and locations. Return an empty list if none are written.\n"
    "- category: the single best job family from the allowed list.\n"
    "- experience_min_years and experience_max_years: required years of experience as integers, -1 when not stated.\n"
)
# Unchanged from prompt p2 so cached responses stay valid; the category answer is no longer used (roles decide it)
RESPONSE_FORMAT = json.dumps({
    "type": "json_schema",
    "json_schema": {
        "name": "job_enrichment",
        "schema": {
            "type": "object",
            "properties": {
                "skills": {"type": "array", "items": {"type": "string"}},
                "category": {"type": "string", "enum": CATEGORY_NAMES},
                "experience_min_years": {"type": "integer"},
                "experience_max_years": {"type": "integer"},
            },
            "required": ["skills", "category", "experience_min_years", "experience_max_years"],
            "additionalProperties": False,
        },
        "strict": True,
    },
})
assert "'" not in RESPONSE_FORMAT, "response format must not contain single quotes"
LLM_SCHEMA = "struct<skills:array<string>, category:string, experience_min_years:int, experience_max_years:int>"

# LLM skills that are too vague to be useful
GENERIC_SKILLS = [
    "communication", "communication skills", "written communication", "verbal communication", "teamwork", "leadership",
    "problem solving", "problem-solving", "analytical skills", "attention to detail", "documentation", "processes",
    "procedures", "policies", "analytics", "implementation", "reporting", "management",
    "stakeholder management", "time management", "collaboration", "ai", "artificial intelligence",
    "mobile", "cloud", "email", "word", "outlook", "ms office", "microsoft office", "ms office suite", "scripting",
]

llm_calls = llm_failures = llm_applied = llm_cache_hits = 0

with task_run("enrich_llm") as metrics:
    if not LLM_ENABLED:
        metrics["message"] = "llm disabled"
    else:
        pending = spark.table(ENRICHED_TABLE).where(F.col("llm_status").isin("pending", "failed")).select("job_key", "description_hash")
        this_prompt = (F.col("endpoint") == LLM_ENDPOINT) & (F.col("prompt_version") == PROMPT_VERSION)

        # 1. Call the LLM only for pending rows without a cached successful response for this endpoint + prompt
        cached_ok = (
            spark.table(LLM_LOG_TABLE).where(this_prompt & F.col("error").isNull() & F.col("response").isNotNull())
            .select("job_key", "description_hash").distinct()
        )
        llm_input = (
            pending.join(cached_ok, ["job_key", "description_hash"], "left_anti")
            .join(
                spark.table(SILVER_TABLE).select(
                    "job_key", "description_hash", "title", "last_seen_at",
                    F.substring(F.coalesce("description", F.lit("")), 1, LLM_MAX_CHARS).alias("description"),
                ),
                ["job_key", "description_hash"],
            )
            .orderBy(F.col("last_seen_at").desc())
            .limit(LLM_MAX_ROWS)
        )
        requests = llm_input.withColumn(
            "request",
            F.concat(F.lit(LLM_PROMPT), F.lit("\nAllowed categories: " + ", ".join(CATEGORY_NAMES)),
                     F.lit("\n\nJob title: "), F.col("title"), F.lit("\n\nJob description:\n"), F.col("description")),
        )
        responses = requests.select(
            "job_key", "description_hash",
            F.expr(f"ai_query('{LLM_ENDPOINT}', request, responseFormat => '{RESPONSE_FORMAT}', failOnError => false)").alias("r"),
        )
        result_field, error_field = ai_result_fields(responses, "r")

        # Persist responses first so the (billable) calls are executed exactly once
        (
            responses.select(
                "job_key", "description_hash", F.lit(LLM_ENDPOINT).alias("endpoint"), F.lit(PROMPT_VERSION).alias("prompt_version"),
                F.col(f"r.{result_field}").cast("string").alias("response"),
                (F.col(f"r.{error_field}").cast("string") if error_field else F.lit(None).cast("string")).alias("error"),
                F.current_timestamp().alias("called_at"), F.lit(JOB_RUN_ID).alias("_job_run_id"),
            )
            .write.mode("append").saveAsTable(LLM_LOG_TABLE)
        )
        new_calls = spark.table(LLM_LOG_TABLE).where((F.col("_job_run_id") == JOB_RUN_ID) & this_prompt)
        llm_calls = new_calls.count()
        llm_failures = new_calls.where(F.col("error").isNotNull() | F.col("response").isNull()).count()

        # 2. Apply the best response per pending row: newest successful one (cached or new), else this run's failure
        best_first = Window.partitionBy("job_key", "description_hash").orderBy(
            (F.col("error").isNull() & F.col("response").isNotNull()).desc(), F.col("called_at").desc()
        )
        calls = (
            spark.table(LLM_LOG_TABLE).where(this_prompt)
            .join(pending, ["job_key", "description_hash"])
            .withColumn("_rank", F.row_number().over(best_first)).where("_rank = 1").drop("_rank")
            .withColumn("parsed", F.from_json("response", LLM_SCHEMA))
            .withColumn("ok", F.col("error").isNull() & F.col("parsed").isNotNull())
        )
        llm_cache_hits = calls.where(F.col("ok") & (F.col("_job_run_id") != JOB_RUN_ID)).count()

        alias_map = F.broadcast(
            spark.table(table("ops", "ref_skills"))
            .select("skill", "skill_group", F.explode("aliases").alias("alias"))
            .select(F.lower("alias").alias("alias"), F.col("skill").alias("canonical_skill"), "skill_group")
            .dropDuplicates(["alias"])
        )
        source_text = spark.table(SILVER_TABLE).select(
            "job_key", "description_hash", F.lower(F.concat_ws(" ", "title", "description")).alias("source_text")
        )
        # Skills not in the taxonomy must look technical: contain a capital, digit or symbol, and not be one plain
        # capitalised word ("Drive", "Excellence" from company values boilerplate)
        looks_technical = F.col("raw_skill").rlike(r"[A-Z0-9+#./&]") & ~F.col("raw_skill").rlike(r"^[A-Z][a-z]+$")
        llm_skills = (
            calls.where("ok")
            .select("job_key", "description_hash", F.posexplode("parsed.skills").alias("pos", "raw_skill"))
            .withColumn("raw_skill", F.trim(F.regexp_replace("raw_skill", r"\s+", " ")))
            .where(F.length("raw_skill").between(2, 30) & (F.size(F.split("raw_skill", " ")) <= 4))
            .where(~F.lower("raw_skill").isin(GENERIC_SKILLS))
            .join(source_text, ["job_key", "description_hash"])
            .where(F.instr("source_text", F.lower("raw_skill")) > 0)  # grounding: must appear in the job text
            .join(alias_map, F.lower("raw_skill") == F.col("alias"), "left")
            .where(F.col("canonical_skill").isNotNull() | looks_technical)
            .groupBy("job_key", "description_hash")
            .agg(
                F.array_distinct(F.collect_list(F.coalesce("canonical_skill", "raw_skill"))).alias("llm_skills"),
                F.array_distinct(F.collect_list(F.coalesce("skill_group", F.lit("Other")))).alias("llm_groups"),
            )
        )
        (
            calls.join(llm_skills, ["job_key", "description_hash"], "left")
            .select(
                "job_key", "description_hash", "ok", "llm_skills", "llm_groups",
                F.when(F.col("parsed.experience_min_years").between(0, 40), F.col("parsed.experience_min_years")).alias("llm_min"),
                F.when(F.col("parsed.experience_max_years").between(0, 40), F.col("parsed.experience_max_years")).alias("llm_max"),
            )
            .createOrReplaceTempView("llm_updates")
        )
        years_to_level = """CASE WHEN {v} IS NULL THEN NULL WHEN {v} < 1 THEN 'Entry' WHEN {v} < 3 THEN 'Junior'
                            WHEN {v} < 6 THEN 'Mid' WHEN {v} < 10 THEN 'Senior' ELSE 'Lead/Manager' END"""
        merge_result = spark.sql(f"""
            MERGE INTO {ENRICHED_TABLE} t
            USING llm_updates s
            ON t.job_key = s.job_key AND t.description_hash = s.description_hash
            WHEN MATCHED AND NOT s.ok THEN UPDATE SET t.llm_status = 'failed', t._job_run_id = '{JOB_RUN_ID}'
            WHEN MATCHED THEN UPDATE SET
              t.skills = slice(array_distinct(concat(coalesce(t.skills, CAST(array() AS ARRAY<STRING>)),
                                                     coalesce(s.llm_skills, CAST(array() AS ARRAY<STRING>)))), 1, 30),
              t.skill_groups = array_distinct(concat(coalesce(t.skill_groups, CAST(array() AS ARRAY<STRING>)),
                                                     coalesce(s.llm_groups, CAST(array() AS ARRAY<STRING>)))),
              t.experience_min_years = coalesce(t.experience_min_years, s.llm_min),
              t.experience_max_years = coalesce(t.experience_max_years, s.llm_max),
              t.experience_text = coalesce(t.experience_text, CASE WHEN s.llm_min IS NOT NULL THEN 'llm' END),
              t.experience_level = coalesce(t.experience_level, {years_to_level.format(v='coalesce(t.experience_min_years, s.llm_min)')}),
              t.llm_status = 'done',
              t.enriched_at = current_timestamp(),
              t._job_run_id = '{JOB_RUN_ID}'
        """).first().asDict()
        llm_applied = int(merge_result.get("num_updated_rows", 0))

        metrics.update(rows_in=llm_calls, rows_out=llm_applied, rows_rejected=llm_failures)
        metrics["details"].update(endpoint=LLM_ENDPOINT, prompt_version=PROMPT_VERSION, max_rows=LLM_MAX_ROWS, cache_hits=llm_cache_hits)

print(f"skills LLM new calls={llm_calls} failures={llm_failures} cache hits={llm_cache_hits} applied={llm_applied}")

In [ ]:
stop_if_job()  # scheduled runs end here; the cells below are for manual inspection

## Inspect

In [ ]:
enriched = spark.table(ENRICHED_TABLE)
display(
    enriched.where("gold_eligible")
    .groupBy("role_method", "category_method", "llm_status").count()
    .orderBy("role_method", "category_method", "llm_status")
)

In [ ]:
display(
    enriched.where("gold_eligible")
    .groupBy("category", "role_title")
    .agg(F.count("*").alias("jobs"), F.round(F.avg("role_score"), 3).alias("avg_score"),
         F.round(F.avg(F.size("skills")), 1).alias("avg_skills"))
    .orderBy("category", F.col("jobs").desc())
)

In [ ]:
display(
    enriched.join(spark.table(SILVER_TABLE).select("job_key", "source", "company_name", "title"), "job_key")
    .where("gold_eligible AND role_title IS NOT NULL")
    .select("source", "company_name", "title", "role_title", "role_score", "role_alternative", "role_alternative_score",
            "role_method", "category", "experience_level", "experience_min_years", F.size("skills").alias("n_skills"), "skills")
    .orderBy(F.rand(seed=11))
    .limit(30)
)

In [ ]:
display(
    enriched.where("gold_eligible AND role_method IS NOT NULL")
    .select(F.round(F.col("role_score") - F.col("role_alternative_score"), 2).alias("margin"))
    .groupBy("margin").count().orderBy("margin")
)

In [ ]:
display(enriched.groupBy("gold_exclusion_reason").count().orderBy(F.col("count").desc()))

In [ ]:
display(spark.table(table("ops", "pipeline_runs")).where("task LIKE 'enrich%'").orderBy(F.col("started_at").desc()).limit(6))